# MIRAGE Quantum Machine Learning for Traffic Classification
di Mario Gabriele Carofano

### Panoramica

### Dataset

### Output


---

In [ ]:
#	LIBRARIES
#   ####################################################################    #

# Importing constant values
import constants

# Data loading and saving
import pickle
import os

# Data manipulation and analysis
import numpy as np
import pandas as pd
from collections import Counter

# Data preprocessing and evaluation
from preprocessing_functions import *
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score

# Machine Learning and Quantum ML
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim
import pennylane as qml

# Data visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Other utilities
import random
import time
import copy

In [ ]:
#	MACROS
#   ####################################################################    #

import importlib
importlib.reload(constants)
from constants import RANDOM_SEED

#   ####################################################################    #

# 1. Configurazione del seed per la riproducibilità.
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
os.environ['PYTHONHASHSEED'] = str(RANDOM_SEED) # Utile per hash di dizionari/set

# 2. Configurazione di PyTorch (CPU e GPU)
torch.manual_seed(RANDOM_SEED)
torch.cuda.manual_seed(RANDOM_SEED)
torch.cuda.manual_seed_all(RANDOM_SEED)

# 3. Configurazione di PyTorch per la riproducibilità su GPU (CUDNN)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# 4. Configurazione del dispositivo (CPU o GPU)
DEVICE = 'mps' if torch.backends.mps.is_available() else 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {DEVICE}")
if DEVICE == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")

---

## Caricamento dei dati

Caricamento del file pickle preprocessato contenente biflussi e etichette.

In [ ]:
import importlib
importlib.reload(constants)
from constants import DATA_PATH

#   ####################################################################    #

print("Loading dataset...")

# Il file pickle contiene due oggetti salvati in sequenza:
# 1. X_raw : i dati numerici dei flussi di traffico, in formato numpy array.
# 2. y_raw : le etichette corrispondenti ai flussi.
# Ogni chiamata a pickle.load() legge il successivo oggetto nel file.
with open(DATA_PATH, "rb") as f:
    X_raw = np.array(pickle.load(f), dtype=np.float32)
    y_raw = np.array(pickle.load(f))

print(f"Data shape: {X_raw.shape}")
print(f"Labels shape: {len(y_raw)}")

# Si definisce un campione d'esempio per tutto il notebook.
example_sample = 7000
print("\nExample Sample (First 5 packets):\n", X_raw[example_sample][:5])
print("Example Label:", y_raw[example_sample])

## Label Encoding

Trasformazione delle etichette categoriche in valori numerici utilizzando `LabelEncoder`.

Ogni classe di traffico viene associata a un identificativo numerico, facilitando l'utilizzo nei modelli ML. Le classi originali vengono memorizzate in `DATASET_CLASSES` per riferimenti futuri, mentre il numero totale di classi viene salvato in `NUM_CLASSES`.

In [ ]:
le = LabelEncoder()

y_encoded = le.fit_transform(y_raw)

DATASET_CLASSES = le.classes_
""" List of unique classes in the dataset, determined by the unique labels in y_raw after encoding. """

NUM_CLASSES = len(le.classes_)
""" Number of unique classes in the dataset. """

print(
    f"Number of classes: {NUM_CLASSES} \n\n" +
	f"Classes: {DATASET_CLASSES}"
)

## Train-Validation-Test split

Il dataset viene suddiviso in tre insiemi distinti mediante doppio split stratificato, le cui dimensioni sono specificate dalle costanti `TRAIN_SIZE`, `VAL_SIZE` e `TEST_SIZE`, preservando la distribuzione originale delle classi.

Se il training set supera `NEW_TRAIN_SIZE` campioni, viene ridotto tramite campionamento casuale per contenere i tempi di addestramento.

In [ ]:
import importlib
importlib.reload(constants)
from constants import (
	TRAIN_SIZE, VAL_SIZE, TEST_SIZE, NEW_TRAIN_SIZE,
    RANDOM_SEED
)

#   ####################################################################    #

# Verifica che le proporzioni di Train, Val e Test sommino a "1".
assert TRAIN_SIZE + VAL_SIZE + TEST_SIZE == 1.0, "Train, Val, Test sizes must sum to 1."

# Il primo split divide il dataset in due parti: Train+Val e Test.
X_temp, X_test, y_temp, y_test = train_test_split(
    X_raw, y_encoded,
    test_size=TEST_SIZE,
    stratify=y_encoded,
    random_state=RANDOM_SEED
)

# Il secondo split divide il Train+Val rimanente in due insiemi: Train e Val.
tmp_size = VAL_SIZE / (TRAIN_SIZE + VAL_SIZE)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp,
    test_size=tmp_size,
    stratify=y_temp,
    random_state=RANDOM_SEED
)

print(
	f"Train shape: {X_train.shape} ({len(X_train)/len(X_raw):.1%}) \n" +
	f"Val shape:   {X_val.shape}   ({len(X_val)/len(X_raw):.1%}) \n" +
	f"Test shape:  {X_test.shape}  ({len(X_test)/len(X_raw):.1%}) \n"
)

# Riduzione della dimensione del training set, se necessario.
if len(X_train) > NEW_TRAIN_SIZE:
    
	# Calcola la frazione del training set da mantenere per ridurlo a NEW_TRAIN_SIZE
	percentage = NEW_TRAIN_SIZE / len(X_train)
    
	X_train, _, y_train, _ = train_test_split(
		X_train, y_train,
		train_size=percentage,
		stratify=y_train,
		random_state=RANDOM_SEED
	)
      
	print(f"Reduced Train shape: {X_train.shape} ({len(X_train)/len(X_raw):.1%})")
      
else:
    
    print("Training set size is less than or equal to the new size. No reduction applied.")
    
	# end if

---

## Preprocessing

In questa sezione vengono raccolte le principali strategie di preprocessing applicate ai biflussi del dataset prima della fase di training. L'obiettivo è trasformare i dati grezzi in una rappresentazione più adatta ai modelli classici e ibridi quantistici. Queste strategie consentono di confrontare diverse modalità di preparazione dei dati, da approcci più semplici e generici a soluzioni più mirate al dominio del traffico di rete.

Le feature originali considerate sono:

- `DIR`: direzione del pacchetto
- `PL`: packet length
- `TCPWIN`: finestra TCP
- `IAT`: inter-arrival time

**Strategies**

**1. Masking del padding + Log1p normalization:**
Approccio guidato dal dominio applicativo. I pacchetti di padding vengono identificati e azzerati per evitare che influenzino il modello. Successivamente viene applicata una trasformazione `Log1p` alle feature numeriche (`PL`, `TCPWIN`, `IAT`) per comprimere il range dinamico e ridurre l'effetto degli outlier.

**2. Min-Max Scaling standard:**
Approccio generico che applica una normalizzazione lineare nell'intervallo `[0, 1]` a tutte le feature. Non gestendo esplicitamente il padding, questa strategia può trattare i valori fittizi come dati reali e risultare sensibile agli outlier.

**3. Fusione DIR/PL + Min-Max Scaling:**
La direzione (`DIR`) e la lunghezza (`PL`) vengono combinate in una singola feature con segno, così da rappresentare il traffico in modo più compatto. Dopo questa trasformazione, il numero di feature passa da 4 a 3 e viene applicato un Min-Max Scaling standard.

**4. Masking del padding + Log1p + fusione DIR/PL:**
Strategia ibrida che unisce i vantaggi della Strategy 1 e della Strategy 3. Prima gestisce correttamente il padding e applica `Log1p`, poi combina `DIR` e `PL` in una feature con segno. In questo modo si ottiene una rappresentazione più compatta, semanticamente coerente e generalmente più robusta.

In [ ]:
import importlib
importlib.reload(constants)
from constants import PREPROCESSING_STRATEGY

#   ####################################################################    #

strategy_map = {
	1: log1pPreprocessing,
	2: minMaxPreprocessing,
	3: lambda X: minMaxPreprocessing(X, combine_dir_pl_flag=True),
	4: lambda X: log1pPreprocessing(X, combine_dir_pl_flag=True),
}

if PREPROCESSING_STRATEGY not in strategy_map:
	raise ValueError("PREPROCESSING_STRATEGY deve essere uno tra: 1, 2, 3, 4")

preprocess_fn = strategy_map[PREPROCESSING_STRATEGY]

print(f"Applying preprocessing strategy {PREPROCESSING_STRATEGY}...")

X_train_proc = preprocess_fn(X_train)
X_val_proc = preprocess_fn(X_val)
X_test_proc = preprocess_fn(X_test)

# Aggiorna il numero di feature in base alla strategia scelta.
N_FEATURES = X_train_proc.shape[2]

print("\nPreprocessing complete.")

print(f"\nTrain shape: {X_train_proc.shape}")
print(f"Val shape:   {X_val_proc.shape}")
print(f"Test shape:  {X_test_proc.shape}")

print(f"\nAdjusted N_FEATURES: {N_FEATURES}")

---

## Dataset & Dataloader

Pytorch Dataset wrapper.

In [ ]:
import importlib
importlib.reload(constants)
from constants import BATCH_SIZE

#   ####################################################################    #

class MirageDataset(Dataset):
	""" ... """
		
	def __init__(self, X, y):
		# X shape: (N, 36, 4)
		# CNN 1D expects (N, Channels, Length) -> (N, 4, 36)

		# Remove to(DEVICE) if dataset doesn't fit in memory
		self.X = torch.FloatTensor(X).permute(0, 2, 1).double().to(DEVICE)

		# Remove to(DEVICE) if dataset doesn't fit in memory
		self.y = torch.LongTensor(y).to(DEVICE)

	def __len__(self):
		return len(self.y)

	def __getitem__(self, idx):
		return self.X[idx], self.y[idx]

train_dataset = MirageDataset(X_train_proc, y_train)
val_dataset = MirageDataset(X_val_proc, y_val)
test_dataset = MirageDataset(X_test_proc, y_test)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True, num_workers=0
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False, num_workers=0
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False, num_workers=0
)

## Model Selection

### Amplitude Embedding Model
A hybrid neural network that leverages Amplitude Embedding to encode classical data into the amplitudes of the quantum state. This approach maximizes data density, allowing the encoding of $2^N$ features into $N$ qubits, preceded by a sigmoid-activated classical dense layer.

In [ ]:
from nn_models import AmpHybridModel as HybridModel

### Angle Embedding Model
A streamlined hybrid model utilizing Angle Embedding, where input features are mapped directly to qubit rotation angles in a 1:1 ratio. It features a classical pre-processing layer followed by a quantum circuit with strongly entangling layers, offering a shallow and noise-resilient embedding strategy.

In [ ]:
from nn_models import AngleHybridModel as HybridModel

### Ring Model
A hybrid architecture that employs a custom Ring Embedding strategy, splitting the input into two sets of features encoded via rotations and circular CNOT entangling patterns. This design doubles the data capacity compared to simple angle embedding and introduces correlations between qubits early in the circuit.

In [ ]:
from nn_models import RingHybridModel as HybridModel

### Waterfall Model
A complex hybrid model featuring a Waterfall Embedding scheme that splits inputs into Y-rotation and Z-rotation blocks. It incorporates a dense, all-to-all "waterfall" connectivity of CNOT gates in the first stage, creating a highly entangled state before the variational ansatz layers.

In [ ]:
from nn_models import WaterfallHybridModel as HybridModel

### Classical 1D CNN Model

In [ ]:
from nn_models import TrafficCNN as HybridModel

### Complex Hybrid Models


##### AmpCnn Model

In [ ]:
from complex_hybrid_models import AmpCnn as HybridModel

##### Classical Twin Model


In [ ]:
from complex_hybrid_models import ClassicalTwinModel as HybridModel

##### Classical Light Model


In [ ]:
from complex_hybrid_models import ClassicalLight as HybridModel

##### CnnAmpCnn Model


In [ ]:
from complex_hybrid_models import CnnAmpCnn as HybridModel

##### Dense Model


In [ ]:
from complex_hybrid_models import Dense as HybridModel

Model instantiation

In [ ]:
import importlib
importlib.reload(constants)
from constants import (
    N_QUBITS, N_LAYERS,
    N_FEATURES, N_PACKETS
)

#   ####################################################################    #

model = HybridModel(
    n_qubits=N_QUBITS,
    n_layers=N_LAYERS,
    n_features=N_FEATURES,
    n_packets=N_PACKETS,
    num_classes=NUM_CLASSES
)

# Sposta il modello sul device e convertilo in double (float64) per precisione quantistica
model = model.to(DEVICE).double()
print(model.get_model_name())
print(model)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Totale parametri addestrabili: {total_params:,}")

## Training Setup

In [ ]:
import importlib
importlib.reload(constants)
from constants import LEARNING_RATE

#   ####################################################################    #

# Configurazione Optimizer
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)


def train_epoch(model, loader, criterion, optimizer):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for inputs, labels in loader:
        # inputs, labels = inputs.to(DEVICE), labels.to(DEVICE) # Remove comment if dataset does't fit into memory

        optimizer.zero_grad()
        outputs = model(inputs)
        # loss = criterion(torch.log(outputs + 1e-10), labels)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        # scheduler.step()

        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

    return running_loss / len(loader), 100. * correct / total

def evaluate(model, loader, criterion):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for inputs, labels in loader:
            # inputs, labels = inputs.to(DEVICE), labels.to(DEVICE) # Remove comment if dataset does't fit into memory
            outputs = model(inputs)
            loss = criterion(outputs, labels)

            running_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

    return running_loss / len(loader), 100. * correct / total

### CrossEntropy Loss function

The cross-entropy loss (or log loss) is a fundamental metric in classification models, measuring the difference between the predicted probability distribution and the true distribution of the labels. It heavily penalizes predictions that are confidently wrong, guiding the model to improve its predictions by minimizing this difference during training.

In [ ]:
criterion = nn.CrossEntropyLoss()

### Wieghted CrossEntropy Loss function

Weighted CrossEntropy assigns different weights to each class based on their frequency in the training set. Classes with fewer samples receive higher weights, helping the model to pay more attention to them during training.

In [ ]:
def compute_class_weights():
    class_counts = Counter(y_train)
    total_samples = sum(class_counts.values())
    weights = []
    for i in range(NUM_CLASSES):
        count = class_counts.get(i, 0)
        if count > 0:
            weights.append(total_samples / (NUM_CLASSES * count))
        else:
            weights.append(1.0)
    return torch.FloatTensor(weights)

In [ ]:
class_weights = compute_class_weights().to(DEVICE).double()
criterion = nn.CrossEntropyLoss(weight=class_weights)

### Focal Loss function

Focal loss is designed to address class imbalance by down-weighting easy examples and focusing more on hard, misclassified examples. Weights are calculated in the same way as Weighted CrossEntropy. Gamma is a tunable focusing parameter that adjusts the rate at which easy examples are down-weighted.

In [ ]:
GAMMA = 2.0 # Focusing parameter. 2 is a common choice.
ALPHA = compute_class_weights().to(DEVICE).double()

Another strategy. Gamma is set to 3.0 to increase the focus on hard examples and ALPHA is set to uniform weights.

In [ ]:
GAMMA = 3.0
ALPHA = torch.ones(NUM_CLASSES, dtype=torch.double).to(DEVICE)

In [ ]:
# Loading Focal loss from torch hub
focal_loss = torch.hub.load(
	'adeelh/pytorch-multi-class-focal-loss',
	model='focal_loss',
	alpha=ALPHA,
	gamma=GAMMA,
	reduction='mean',
	device=DEVICE,
	dtype=torch.double,
	force_reload=False
)

criterion = focal_loss

## Training Loop

In [ ]:
# Dizionario per salvare la history
history = {'accuracy': [], 'val_accuracy': [], 'loss': [], 'val_loss': []}

In [ ]:
print(f"[INFO] Addestramento iniziato. {model.get_model_name()}")
start_time = time.time()

best_val_loss = float('inf')
best_model_wts = copy.deepcopy(model.state_dict())
patience_counter = 0

EPOCHS = 50
PATIENCE = 10
EARLY_STOPPING = True

for epoch in range(EPOCHS):
    start_epoch_time = time.time()
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer)
    val_loss, val_acc = evaluate(model, val_loader, criterion)
    end_epoch_time = time.time()
    epoch_duration = end_epoch_time - start_epoch_time

    history['loss'].append(train_loss)
    history['accuracy'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_accuracy'].append(val_acc)

    print(f"Epoch {epoch+1}/{EPOCHS} | "
          f"Loss: {train_loss:.4f} - Acc: {train_acc:.4f} | "
          f"Val Loss: {val_loss:.4f} - Val Acc: {val_acc:.4f}"
          f" | Time: {epoch_duration:.2f}s")


    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_model_wts = copy.deepcopy(model.state_dict())
        patience_counter = 0
        print(f"  -> Validation loss improved. Model saved.")
    else:
        patience_counter += 1
        print(f"  -> No improvement. Patience: {patience_counter}", f"/ {PATIENCE}" if EARLY_STOPPING else "")

    if patience_counter >= PATIENCE and EARLY_STOPPING:
        print("Early stopping triggered.")
        break

total_time = time.time() - start_time
print(f"\nTraining complete in {total_time/60:.2f} minutes.")

# Load best model weights
last_model = copy.deepcopy(model)
model.load_state_dict(best_model_wts)

Saving Results

In [ ]:
# Saving results
model_name = "50E_log1p_Dense"
output_dir = "models/crossentropy/cross_AMPcnn_quantum_models"

os.makedirs(f"{output_dir}", exist_ok=True)
os.makedirs(f"{output_dir}/{model_name}", exist_ok=True)

# with open(f"{output_dir}/{model_name}/model_summary.txt", "w") as f:
#     model.summary(print_fn=lambda x: f.write(x + "\n"))

df_history = pd.DataFrame(history)
df_history.to_csv(f"{output_dir}/{model_name}/training_history.csv", index=False)

torch.save(model.state_dict(), f"{output_dir}/{model_name}/model.pth")

## Testing and Evaluation

(Optnional) Load Model

In [ ]:
output_dir = "models/crossentropy/cross_AMPcnn_quantum_models"
model_name = "50E_log1p_CnnAmpCnn"

model = HybridModel(
    n_qubits=N_QUBITS,
    n_layers=N_LAYERS,
    n_features=N_FEATURES,
    n_packets=N_PACKETS,
    num_classes=NUM_CLASSES
)
model = model.double()
weights = torch.load(f"{output_dir}/{model_name}/model.pth", map_location=DEVICE)
load_result = model.load_state_dict(weights)
print(f"Esito caricamento: {load_result}")
model = model.to(DEVICE)
model.eval()
print('Model loaded correctly')


# Load training history from csv
df_history = pd.read_csv(f"{output_dir}/{model_name}/training_history.csv")
history = df_history.to_dict(orient='list')


Testing set loss and accuracy

In [ ]:
test_loss, test_acc = evaluate(model, test_loader, criterion)
print(f"Test Loss: {test_loss:.4f} | Test Acc: {test_acc:.2f}%")

In [ ]:
acc = history['accuracy']
val_acc = history['val_accuracy']
loss = history['loss']
val_loss = history['val_loss']
epochs_range = range(len(acc))

training_validation_plots = plt.figure(figsize=(12, 5))

# Plot Loss
plt.subplot(1, 2, 1)
plt.plot(epochs_range, loss, label='Train Loss')
plt.plot(epochs_range, val_loss, label='Val Loss')
plt.title('Loss over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

# Plot Accuracy
plt.subplot(1, 2, 2)
plt.plot(epochs_range, acc, label='Train Acc')
plt.plot(epochs_range, val_acc, label='Val Acc')
plt.title('Accuracy over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

plt.show()

In [ ]:
# salva plot su file
training_validation_plots.savefig(f"{output_dir}/{model_name}/training_validation_plots.png")

In [ ]:
# Confusion Matrix
model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for inputs, labels in test_loader:
        inputs = inputs.to(DEVICE)
        outputs = model(inputs)
        _, predicted = outputs.max(1)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy()) # Replace with all_labels.extend(labels.numpy()) if using CPU

In [ ]:
cm = confusion_matrix(all_labels, all_preds, normalize='true')
confusion_matrix_plot = plt.figure(figsize=(10, 8))


mask = cm == 0
ax =sns.heatmap(
    cm,
    annot=False,
    fmt='',
    cmap='plasma_r',                 # Mappa colori plasma invertita
    linewidths=0.5,                  # Spessore linee della griglia
    mask=mask,                       # Applica la maschera
    linecolor='black',               # Colore linee della griglia
    square=True,                     # Celle quadrate
    cbar_kws={"ticks": [0.1, 1, 10, 100]}, xticklabels=le.classes_, yticklabels=le.classes_)
ax.set_facecolor('white')

empty_cols = np.where(cm.sum(axis=0) == 0)[0]

for col in empty_cols:
    for row in range(cm.shape[0]):
        # col + 0.5 e row + 0.5 servono a centrare il testo nella cella
        ax.text(col + 0.5, row + 0.5, '●',
                ha='center', va='center',
                color='black', fontsize=15)


plt.title("Confusion Matrix")
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.show()

In [ ]:
# salva plot su file
confusion_matrix_plot.savefig(f"{output_dir}/{model_name}/confusion_matrix.png")

In [ ]:
# --- 3. Classification Report ---
print("Classification Report:")
report_dict = classification_report(all_labels, all_preds, target_names=le.classes_, labels=np.arange(40), digits=4, zero_division=0)
print(report_dict)

In [ ]:
# Salva classification report su file
with open(f"{output_dir}/{model_name}/classification_report.txt", "w") as f:
    f.write(report_dict)
